# RQ3 — Recruitment in CAG10/CAG47 mixtures

Examine how short CAG10 RNAs associate with longer CAG47 assemblies.

**Important:** whole-system mean degree is not the mean internal degree of
the **largest cluster**. For largest-cluster connectivity, always use
`largest_mean_degree` and `largest_n_edges`.

In [ ]:
from pathlib import Path
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Works when Jupyter starts from the repository or analysis/notebooks/.
here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
OUTPUT_ROOT = ROOT / "analysis_results"


In [ ]:
from src.plots.figures import (
    plot_largest_contact_composition, plot_mixture_ranked_bars,
    plot_metric_series, plot_ranked_clusters
)
DENSITY = 0.01
STIFFNESS = 2.5
MIXTURE_SWEEP = "52b_from_frame"
LENGTH_SWEEP = "51a_repeat_number"
mixture = filter_rows(
    load_sweep(OUTPUT_ROOT, MIXTURE_SWEEP, "summary"),
    density=DENSITY, angle_k_ppp=STIFFNESS
)
pure = filter_rows(
    load_sweep(OUTPUT_ROOT, LENGTH_SWEEP, "summary"),
    density=DENSITY, angle_k_ppp=STIFFNESS
)
print(f"Mixture conditions: {len(mixture)}; pure reference conditions: {len(pure)}")

## Ranked clusters: pure systems and species-resolved mixture

These are **particle fractions of the entire simulation**, not fractions
within the cluster. CAG10 and CAG47 stacked material sums to each ranked
category. Pure references are derived from the RQ2 sweep at the same density
and analysis window.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2), layout="constrained")
plot_ranked_clusters(filter_rows(pure, repeat_count=10), axes[0], style=STYLE)
axes[0].set_title("Pure (CAG)10")
plot_ranked_clusters(filter_rows(pure, repeat_count=47), axes[1], style=STYLE)
axes[1].set_title("Pure (CAG)47")
plot_mixture_ranked_bars(mixture, axes[2], style=STYLE, annotate_molecules=True)
axes[2].set_title("CAG10 + CAG47: ranked material")
fig.suptitle("RQ3 · ranked-cluster composition")
plt.show()

## Intermolecular contacts INSIDE the largest cluster

Three contact classes: 10–10, 10–47, and 47–47. The stack is normalized
to **intermolecular particle contacts within that component only**;
the annotations are mean absolute contact counts.

In [ ]:
fig, axes = plt.subplots(figsize=(8, 4.5), layout="constrained")
plot_largest_contact_composition(mixture, axes, style=STYLE, annotate_counts=True)
axes.set_title("Largest-cluster contact composition")
plt.show()

## Largest-cluster size and connectivity

One panel summarizes material incorporation; the other quantifies qualifying
molecule-pair graph edges *inside the largest component*.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
plot_metric_series(
    mixture, axes[0], metric="largest_fraction_total_particles",
    style=STYLE, band="std", color="#286D9E"
)
plot_metric_series(
    mixture, axes[1], metric="largest_mean_degree",
    style=STYLE, band="std", color="#BF7849"
)
fig.suptitle("Largest cluster: particle fraction and internal mean degree")
plt.show()

## Optional interactive PyVis molecular network

This demonstration uses the cached **molecule graph NPZ**, not a framewise CSV.
Set `EXPORT_PYVIS` to True and select a parameter-run folder from the sweep.
It writes portable HTML locally; the notebook's ordinary figures do not require PyVis.

In [ ]:
EXPORT_PYVIS = False
if EXPORT_PYVIS:
    from src.plots.networks import save_pyvis_network
    # A summary row has no run folder; resolve one from replicates.csv.
    reps = filter_rows(load_sweep(OUTPUT_ROOT, MIXTURE_SWEEP, "replicates"),
                       density=DENSITY, angle_k_ppp=STIFFNESS)
    first = reps.iloc[0]
    parameter_run = OUTPUT_ROOT / MIXTURE_SWEEP / first["folder_name"]
    stem = first["traj_replicate"]
    graph_npz = parameter_run / "computed_metrics" / "mol_graph" / f"{stem}_molecule_contact_edges.npz"
    lookup_npy = parameter_run / "computed_metrics" / "pairing" / "molecule_lookup.npy"
    result = save_pyvis_network(
        graph_npz, lookup_npy,
        OUTPUT_ROOT / MIXTURE_SWEEP / "figures" / "mixture_network.html",
        style=STYLE, species_by_length={30: "cag10", 141: "cag47"}
    )
    print(f"Open interactive network: {result}")


## Further questions

Compare contact-type composition with species-specific recruitment and
saturation metrics. Interactive molecular contact networks can be exported
separately from cached graph NPZs; they are not a dependency of the
CSV-only notebook figures.